__Prerequisites:__

* Access to the [`bdcmsg22/QC_metabolomics`](https://app.terra.bio/#workspaces/bdcmsg22/QC_metabolomics) workspace where the raw metabolomics data is stored.

In [ ]:
library(data.table)
library(matrixStats)
library(gt)
'%ni%' <- Negate('%in%')

options(datatable.na.strings=c('NA',''))

dir.create("data/raw/metabolomics",             rec=T)
dir.create("data/derived/metabolomics/cleaned", rec=T)
dir.create("data/derived/metabolomics/QCd",     rec=T)

aligned_csvs <- list(CP='data/raw/metabolomics/C8P_data.csv',  # C8-positive
                     CN='data/raw/metabolomics/C18N_data.csv', # C18-negative
                     HP='data/raw/metabolomics/HP_data.csv',   # HILIC-positive
                     AN='data/raw/metabolomics/AM_data.csv')   # Amide-negative

# Download
if(!all(file.exists(unlist(aligned_csvs))))
  system("gcloud storage cp gs://fc-secure-4b3e979d-ba8b-43c4-a5ec-b41ab42ce606/aligned/MESA_WHI_FHS/raw_data/* data/raw/metabolomics") # Workspace: bdcmsg22/QC_metabolomics


# MESA-specific metabolomics files. Only actually needed for their sample metadata.
mesa_csvs    <- list(CP='data/raw/metabolomics/23_0207_MESA_Pilot_X01_Broad_C8-pos.csv',    # C8-positive
                     CN='data/raw/metabolomics/23_0210_MESA_Pilot_X01_Broad_C18-neg.csv',   # C18-negative
                     HP='data/raw/metabolomics/23_0207_MESA_Pilot_X01_Broad_HILIC-pos.csv', # HILIC-positive
                     AN='data/raw/metabolomics/23_0210_MESA_Pilot_X01_BIDMC_Amide-neg.csv') # Amide-negative

# Download original xlsx data and convert to csv, faster to work with.
# It's extremely slow to read from xlsx, but this only needs to run once and then we can use the faster csv.
if(!all(file.exists(unlist(mesa_csvs)))) {
  system("gcloud storage cp gs://fc-secure-4b3e979d-ba8b-43c4-a5ec-b41ab42ce606/MesaMetabolomics_PilotX01_2023_02_16/23_02* data/raw/metabolomics")
  mesa_xlsxs <- sub('.csv','.xlsx', mesa_csvs)
  Map(data_csvs,data_xlsxs, f=\(csv,xlsx)
    xlsx |> readxl::read_xlsx(col_names=F) |> fwrite(csv, col.names=F) )
}

# Inspect raw data
There are files for 4 LC-MS methods: C8-positive, C18-negative, HILIC-positive, Amide-negative ([descriptions here](https://topmed.nhlbi.nih.gov/sites/default/files/TOPMed_CORE_Year3_Broad-BIDMC_metabolomics_methods.pdf)).

In [ ]:
quick_look <- \(file) fread(file, nrow=12, select=1:12) |> knitr::kable()
lapply(aligned_csvs, quick_look)
lapply(mesa_csvs, quick_look)

# Extract sample info 
FHS & WHI have separate metabolomics sample metadata files. However, the MESA sample metadata must be extracted.

## FHS & WHI sample info
The only column common to all FHS + MESA + WHI is the `SAMPLE_ID`s. However, FHS + WHI have a couple columns that can be harmonized.

In [ ]:
sample_info_fhs <- fread(cmd='gcloud storage cat gs://fc-secure-4b3e979d-ba8b-43c4-a5ec-b41ab42ce606/metadata/FHS/omics_sample_metadata_FHS_metabolomics.txt')
sample_info_whi <- fread(cmd='gcloud storage cat gs://fc-secure-4b3e979d-ba8b-43c4-a5ec-b41ab42ce606/metadata/WHI/omics_sample_metadata_WHI_metabolomics.csv')

# Harmonize colnames of similar information
lapply(list(sample_info_fhs, sample_info_whi), \(d) setnames(d, skip_absent=T,
  old=c('subject_id', 'Assay_lab', 'Sample_container_ID', 'sample_row_column_ID'),
  new=c('SUBJECT_ID', 'assay_lab', 'sample_box_ID',       'Sample_well_ID'      ))) |> invisible()

# "Broad Institute" -> "Broad" so it is consistent between the two files
sample_info_whi[assay_lab == 'Broad Institute', assay_lab := 'Broad']

sample_info_fhs[, cohort := 'FHS']
sample_info_whi[, cohort := 'WHI']

## MESA sample info

In [ ]:
sample_info_mesa <- list()
sample_info_mesa$CP <- fread(mesa_csvs$CP, nrow=10, drop=1:8) |> transpose()
sample_info_mesa$CN <- fread(mesa_csvs$CN, nrow=10, drop=1:8) |> transpose()
sample_info_mesa$HP <- fread(mesa_csvs$HP, nrow=10, drop=1:8) |> transpose()
sample_info_mesa$AN <- fread(mesa_csvs$AN, nrow= 9, drop=1:8) |> transpose()

names(sample_info_mesa$CP) <- names(sample_info_mesa$CN) <- names(sample_info_mesa$HP) <- c('extr_date','inject_date','batch','inject_order','sample_type',           'TOPMed_ID','checksum','raw_fnm','project','SAMPLE_ID')
names(sample_info_mesa$AN) <-                                                             c(            'inject_date','batch','inject_order','sample_type','BIDMC_Id','TOPMed_ID',           'raw_fnm','project','SAMPLE_ID')

by_cols <- c('SAMPLE_ID','project')
sample_info_mesa <- sample_info_mesa$CP |>
       merge(y=sample_info_mesa$CN, suffixes=c('','_cn'), by=by_cols, all=T) |>
       merge(y=sample_info_mesa$HP, suffixes=c('','_hp'), by=by_cols, all=T) |>
       merge(y=sample_info_mesa$AN, suffixes=c('','_an'), by=by_cols, all=T) |>
       setnames(c('extr_date',   'inject_date',   'batch',   'inject_order',   'sample_type',   'checksum',   'raw_fnm'   ),
                c('extr_date_cp','inject_date_cp','batch_cp','inject_order_cp','sample_type_cp','checksum_cp','raw_fnm_cp'))

#sample_info_mesa[duplicated(SAMPLE_ID),SAMPLE_ID] # None of the duplicated IDs are TOM ids.

# Remove "TOPMed_ID" to reduce confusion. It is simply a subset of SAMPLE_ID, where some of the QC samples are NA.
sample_info_mesa[, TOPMed_ID := NULL]

# Separate amine batch into batch & plate number
sample_info_mesa[, plate_an := sub('B[0-9]+','', batch_an)]
sample_info_mesa[, batch_an := sub('P[0-9]+','', batch_an)]

# Dates were converted to integers in readxl::read_xlsx. This integer represents the # of days after 1899-12-30 (see ?as.Date.)
date_cols <- grep('date', names(sample_info_mesa), val=T)
sample_info_mesa[, (date_cols) := lapply(.SD, \(x) as.numeric(x) |> as.Date(origin='1899-12-30')), .SDcols=date_cols]

sample_info_mesa[, cohort := 'MESA']

## Concatenate & write sample info

In [ ]:
sample_info <- rbindlist(list(sample_info_mesa, sample_info_fhs, sample_info_whi), fill=T)

sample_info[, is_qc_sample := !grepl("TOM",SAMPLE_ID)]

fwrite(sample_info, "data/derived/metabolomics/sample_info-aligned.csv")
gt(head(sample_info)) |> gt:::as.tags.gt_tbl()

# Extract metabolite metadata
Then, create a file mapping these IDs back to their metabolite metadata such as M/Z and RT.

In [ ]:
met_info <- rbindlist(fill=T, list(
  fread(aligned_csvs$CP, select=1: 9),
  fread(aligned_csvs$CN, select=1: 9),
  fread(aligned_csvs$HP, select=1: 9),
  fread(aligned_csvs$AN, select=1:10)[is.na(Metabolite) | !duplicated(Metabolite)] # AM_data.csv has duplicated Metabolite. More details in the "Exctract measurements" code block.
))

# For consistency with MESA met info, Amide-Negative-sMRM -> Amide-neg
met_info[, Method := sub('Amide-Negative-sMRM','Amide-neg',Method)]

# Compound IDs may be duplicated across CP/CN/HP/AN. A concatenated `<ID>_<Method>` variable ensures IDs are unique.
met_info[,                      unique_met_id := paste0(Compound_ID,'_',gsub('-','_',Method))] # R doesn't like '-' in names.
met_info[grepl('Amide',Method), unique_met_id := paste0('met', .I,  '_',gsub('-','_',Method))] # Amines have no Compound_ID, so generate them.
#sum(duplicated(met_info$unique_id)) # No dup IDs!
met_info

fwrite(met_info, "data/derived/metabolomics/met_info-aligned.csv")
gt(head(met_info))

# Extract measurement data
## Fix amine data errors
There are several errors with the amine data:

* 1 strange string value
* 19 negative measurements, which doesn't make sense in the context of metabolite measurements.
* 2 duplicated metabolites, which are identical except for differing measurements in FHS samples.

In [ ]:
data <- list()
data$AN <- fread(aligned_csvs$AN, drop=c(1:5,7,9:10))
# Keep HMDB_ID and Metabolite columns to identify metabolites uniquely, since these weren't given Compound_IDs.

# There's a weird non-numeric string in one of the cells. Set to NA.
data$AN[20,  MESA__TOM148122 ] # "1+2559:2584.03935869994933"
data$AN[20, 'MESA__TOM148122'] <- NA

# There are 19 negative measurements, which doesn't make sense. Set to NA.
data$AN[, sum(data$AN<0, na.rm=T)] # 19
data$AN[data$AN<0] <- NA

# There are nearly-duplicate rows for "1,5-AG / 1-deoxyglucose" and "N-Acetyl-D-Glucosamine".
# The only difference is their value in FHS samples.
# Uncomment if you wish to inspect the non-matching values.
#non_matching_colnames <- data$AN[1:2][, names(.SD)[sapply(.SD, \(x) x[1]!=x[2])]] |> na.exclude()
#row1 <- data$AN[1, .SD, .SDcols=non_matching_colnames] |> unlist()
#row2 <- data$AN[2, .SD, .SDcols=non_matching_colnames] |> unlist()
#plot(row1,row2)
#non_matching_colnames <- data$AN[103:104][, names(.SD)[sapply(.SD, \(x) x[1]!=x[2])]] |> na.exclude()
#row103 <- data$AN[103, .SD, .SDcols=non_matching_colnames] |> unlist()
#row104 <- data$AN[104, .SD, .SDcols=non_matching_colnames] |> unlist()
#plot(row103,row104)

data$AN <- data$AN[!duplicated(Metabolite) | is.na(Metabolite)] # Removing duplicates (arbitrarily chosen)

# Assign the proper metabolite ids we generated in met info
data$AN[
  ][, unique_met_id := met_info[
    ][Method=='Amide-neg'
    ][match(fcoalesce(data$AN$Metabolite,  data$AN$HMDB_ID),
            fcoalesce(        Metabolite,          HMDB_ID))
      , unique_met_id ]
  ][, Metabolite := NULL
  ][, HMDB_ID := NULL
] |> invisible()

# Finally, convert to matrix
data$AN <- data$AN |> as.matrix(rownames='unique_met_id') |> `mode<-`('numeric') |> t()

## Other LC-MS files, and write
The other non-amide methods are more well-behaved.

In [ ]:
data$CP <- fread(aligned_csvs$CP, drop=1:9, colClasses='double') |> t() |> `colnames<-`(paste0(fread(aligned_csvs$CP, select=2)$Compound_ID, '_C8_pos'   ))
data$CN <- fread(aligned_csvs$CN, drop=1:9, colClasses='double') |> t() |> `colnames<-`(paste0(fread(aligned_csvs$CN, select=2)$Compound_ID, '_C18_neg'  ))
data$HP <- fread(aligned_csvs$HP, drop=1:9, colClasses='double') |> t() |> `colnames<-`(paste0(fread(aligned_csvs$HP, select=2)$Compound_ID, '_HILIC_pos'))

data <- lapply(data, \(m) m[grepl('TOM',rownames(m)),] ) # Rm QC samples

Map(fwrite, data, paste0("data/derived/metabolomics/cleaned/",names(data),"_cleaned-aligned.csv"), row.names=T)

In [ ]:
data$CP[1:3,1:3] # Check out the cleaned data!
data$CN[1:3,1:3]
data$HP[1:3,1:3]
data$AN[1:3,1:3]

# QC
1\. Remove signatures w/ variance = 0\
2\. Remove signatures w/ >25% missingness\
3\. Half-min Impute\
4\. Log2 (except the amide data, which is already)\
5\. Winsorize to 5*sd\
Missingness removal is done in two separate steps, where high-missingness metabolites are removed first, _then_ samples (to prioritize not losing samples).\
Batch adjustment is best done at the same time as adjusting for other covariates, later.

In [ ]:
winsorizeColsBySd <- \(m, n_sd) {
  cmeans <- colMeans(m,na.rm=T)
  csds   <- colSds  (m,na.rm=T)
  lower <- cmeans - n_sd*csds
  upper <- cmeans + n_sd*csds
  t(pmin(pmax(t(m),lower),upper)) # lower/upper are vectors of length ncol(m), recycled for each row of the matrix.
}

data <- Map(data, names(data), f=\(d, nm) {
  d <- d[, colVars(d,na.rm=T) > 0] # step 1
  d <- d[,colSums(is.na(d)) < 0.25*nrow(d) ] # step 2a
  d <- d[ rowSums(is.na(d)) < 0.25*ncol(d),] # step 2b
  d[is.na(d)] <- (colMins(d,na.rm=T)/2)[col(d)[is.na(d)]] # step 3
  if(nm!='AN') d <- log2(d+1) # step 4
  d <- winsorizeColsBySd(d,5) # step 5
})

Map(fwrite, data, paste0('data/derived/metabolomics/QCd/',names(data),'_QCd-aligned.csv'))

# Merge CP/CN/HP/AN datasets
8174/12183 non-QC samples have metabolomics data for all of CP/CN/HP/AN. We would prefer to preserve those ~4k samples that don't, so we merge the metabolomics data with `all=T`.

In [ ]:
data_dts <- lapply(data, as.data.table, keep.rownames='TOM_Id')
data_merged <- Reduce(\(x,y) merge(x,y, all=T, by='TOM_Id'), data_dts)
fwrite(data_merged, 'data/derived/metabolomics/QCd/merged_QCd-aligned.csv')

---

# Plots
Plots are made using the CP/HP/AN data separately (not merged).

## Distribution of metabolite medians

In [ ]:
par(mfrow=c(1,4))
Map(data, names(data), f = \(m,nm) hist(colMedians(m), main=nm))

## Skew vs. Kurtosis

In [ ]:
plotSkewKurt <- function(df, title) {
  n <- nrow(df); ms <- colMeans(df); sds <- colSds(df)
  skews <- sapply(1:ncol(df),\(col) (sum((df[,col]-ms[col])^3)/sds[col]^3)/n   )
  kurts <- sapply(1:ncol(df),\(col) (sum((df[,col]-ms[col])^4)/sds[col]^4)/n-3 )
  sk <- data.frame(skews=skews, kurts=kurts)

  {plot(sk$skews, sk$kurts, pch = 16, cex = 0.5, main = title, 
    xlab = "Skews", ylab = "Kurts", col = "black") +
   abline(v = -0.5, lty = 2,  lwd = 2) + 
   abline(v =  0.5, lty = 2,  lwd = 2) +
   abline(h = -2.0, lty = 2,  lwd = 2) +
   abline(h =  2.0, lty = 2,  lwd = 2)}
}

par(mfrow=c(1,4))
Map(plotSkewKurt, data, names(data))

## Individual metabolite distributions
You may need to zoom in all the way, and use Windows Magnifier (Win+"+" keys) to see.

In [ ]:
plotAllMets <- function(mtx) {
  par(mar=c(.1,.1,.4,.1), oma=rep(0,4), xaxt='n', yaxt='n', lwd=0.1,
      mfrow = c(ceiling(ncol(mtx)/30), 30)
  )
  sapply(1:ncol(mtx), FUN = \(i)
    plot(mtx[,i], main=colnames(mtx)[i], cex.main=0.4,
         pch=16, col=rgb(0,0,0,0.5), cex=0.1)
  )
}

png("met_dists_CP.png", width = 10000, height =  9000, res=800)
plotAllMets(data$CP)
dev.off()
png("met_dists_CN.png", width = 10000, height = 14000, res=800)
plotAllMets(data$CN)
dev.off()
png("met_dists_HP.png", width = 10000, height =  8000, res=800)
plotAllMets(data$HP)
dev.off()
png("met_dists_AN.png", width = 10000, height =  1200, res=800)
plotAllMets(data$AN)
dev.off()

![](met_dists_CP.png)

![](met_dists_CN.png)

![](met_dists_HP.png)

![](met_dists_AN.png)